### NOTEBOOK 11 — WEB DEMO DEVELOPMENT

#### Mục tiêu

Xây dựng ứng dụng Web Demo bằng Streamlit để minh họa quy trình phát hiện, định vị Mass và phân loại bệnh lý bằng kiến trúc Two-stage Pipeline (YOLOv5n + InceptionV3).

Ứng dụng sử dụng checkpoint đã huấn luyện, hỗ trợ tải ảnh JPG, PNG, DICOM và hiển thị kết quả trực quan.

Sản phẩm phục vụ mục đích nghiên cứu, thử nghiệm và giáo dục; không thay thế chẩn đoán y khoa.


In [13]:
# 11.1 — Khởi tạo môi trường và cấu hình ứng dụng

from pathlib import Path
import sys
import json
import platform
from datetime import datetime
from importlib import metadata

# 1. CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

YOLO_REPO = PROJECT_ROOT / "external" / "yolov5"

TRAINING_ROOT = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb08_model_training"
)

WEB_DEMO_ROOT = PROJECT_ROOT / "WebDemo"

APP_ROOT = WEB_DEMO_ROOT / "app"
UTILS_ROOT = APP_ROOT / "utils"
ASSETS_ROOT = WEB_DEMO_ROOT / "assets"
CONFIG_ROOT = WEB_DEMO_ROOT / "config"
REPORT_ROOT = WEB_DEMO_ROOT / "reports"
OUTPUT_ROOT = WEB_DEMO_ROOT / "outputs"

# 2. KIỂM TRA ĐƯỜNG DẪN GỐC

print("11.1 — KHỞI TẠO MÔI TRƯỜNG WEB DEMO")
print("-" * 55)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục dự án: {PROJECT_ROOT}"
    )

print(f"Project root: PASS")
print(f"YOLOv5 repository: {'PASS' if YOLO_REPO.is_dir() else 'MISSING'}")
print(f"Training results: {'PASS' if TRAINING_ROOT.is_dir() else 'MISSING'}")

if not TRAINING_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục kết quả huấn luyện: {TRAINING_ROOT}"
    )

# 3. TẠO CẤU TRÚC THƯ MỤC WEB DEMO

folders = [
    WEB_DEMO_ROOT,
    APP_ROOT,
    UTILS_ROOT,
    ASSETS_ROOT,
    CONFIG_ROOT,
    REPORT_ROOT,
    OUTPUT_ROOT,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCẤU TRÚC THƯ MỤC")

for folder in folders:
    print(f"[PASS] {folder.relative_to(PROJECT_ROOT)}")

# 4. KIỂM TRA PHIÊN BẢN PYTHON VÀ THƯ VIỆN

package_mapping = {
    "PyTorch": "torch",
    "Torchvision": "torchvision",
    "Streamlit": "streamlit",
    "NumPy": "numpy",
    "Pandas": "pandas",
    "Pillow": "Pillow",
    "OpenCV": "opencv-python",
    "Matplotlib": "matplotlib",
    "Pydicom": "pydicom",
}

package_versions = {}
missing_packages = []

print("\nKIỂM TRA THƯ VIỆN")

for display_name, package_name in package_mapping.items():
    try:
        version = metadata.version(package_name)
        package_versions[display_name] = version
        print(f"[PASS] {display_name}: {version}")
    except metadata.PackageNotFoundError:
        package_versions[display_name] = None
        missing_packages.append(package_name)
        print(f"[MISSING] {display_name}")

# 5. KIỂM TRA PYTORCH VÀ GPU

torch_info = {
    "available": False,
    "version": None,
    "cuda_available": False,
    "cuda_version": None,
    "gpu_name": None,
}

try:
    import torch

    torch_info["available"] = True
    torch_info["version"] = torch.__version__
    torch_info["cuda_available"] = torch.cuda.is_available()
    torch_info["cuda_version"] = torch.version.cuda

    if torch.cuda.is_available():
        torch_info["gpu_name"] = torch.cuda.get_device_name(0)

    print("\nKIỂM TRA THIẾT BỊ")
    print(f"PyTorch: {torch_info['version']}")
    print(f"CUDA available: {torch_info['cuda_available']}")
    print(f"CUDA version: {torch_info['cuda_version']}")
    print(f"GPU: {torch_info['gpu_name']}")

except ImportError:
    print("\n[MISSING] Không thể import PyTorch.")

# 6. TẠO CẤU HÌNH CƠ BẢN CHO ỨNG DỤNG

app_config = {
    "project_name": "Breast Cancer Detection Demo",
    "author": "Lương Hoàng Yến",
    "dataset": "CBIS-DDSM",
    "architecture": "Two-stage Pipeline",
    "detection_model": "Fine-tuned YOLOv5n 640",
    "classification_model": "InceptionV3",
    "input_formats": ["jpg", "jpeg", "png", "dcm"],
    "max_upload_size_mb": 20,
    "yolo": {
        "image_size": 640,
        "confidence_threshold": 0.20,
        "nms_iou_threshold": 0.60,
    },
    "cnn": {
        "model": "InceptionV3",
        "image_size": 299,
        "classification_threshold": 0.30,
        "transform_input": True,
        "crop_margin": 0.10,
    },
    "paths": {
        "project_root": str(PROJECT_ROOT),
        "yolo_repo": str(YOLO_REPO),
        "training_root": str(TRAINING_ROOT),
        "web_demo_root": str(WEB_DEMO_ROOT),
        "app_root": str(APP_ROOT),
        "utils_root": str(UTILS_ROOT),
        "assets_root": str(ASSETS_ROOT),
        "output_root": str(OUTPUT_ROOT),
    },
}

config_path = CONFIG_ROOT / "app_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        app_config,
        file,
        ensure_ascii=False,
        indent=4,
    )

# 7. LƯU BÁO CÁO MÔI TRƯỜNG

environment_report = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "python_version": sys.version,
    "platform": platform.platform(),
    "package_versions": package_versions,
    "missing_packages": missing_packages,
    "torch": torch_info,
    "paths": {
        "project_root_exists": PROJECT_ROOT.is_dir(),
        "yolo_repo_exists": YOLO_REPO.is_dir(),
        "training_root_exists": TRAINING_ROOT.is_dir(),
        "web_demo_root_exists": WEB_DEMO_ROOT.is_dir(),
    },
    "config_file": str(config_path),
}

report_path = REPORT_ROOT / "nb11_1_environment_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(
        environment_report,
        file,
        ensure_ascii=False,
        indent=4,
    )

# 8. TỔNG KẾT

print("\n" + "-" * 55)
print("KẾT QUẢ KHỞI TẠO")

print(f"Thư mục ứng dụng: {WEB_DEMO_ROOT}")
print(f"File cấu hình: {config_path}")
print(f"Báo cáo môi trường: {report_path}")

if missing_packages:
    print("\nCÁC THƯ VIỆN CÒN THIẾU:")
    for package in missing_packages:
        print(f"- {package}")

    print("\nLệnh cài đặt có thể sử dụng:")
    print(
        "%pip install " + " ".join(missing_packages)
    )
else:
    print("\nTất cả thư viện cần kiểm tra đã được cài đặt.")

print("\nHOÀN TẤT CELL 11.1")


11.1 — KHỞI TẠO MÔI TRƯỜNG WEB DEMO
-------------------------------------------------------
Project root: PASS
YOLOv5 repository: PASS
Training results: PASS

CẤU TRÚC THƯ MỤC
[PASS] WebDemo
[PASS] WebDemo\app
[PASS] WebDemo\app\utils
[PASS] WebDemo\assets
[PASS] WebDemo\config
[PASS] WebDemo\reports
[PASS] WebDemo\outputs

KIỂM TRA THƯ VIỆN
[PASS] PyTorch: 2.5.1+cu121
[PASS] Torchvision: 0.20.1+cu121
[PASS] Streamlit: 1.67.0
[PASS] NumPy: 2.4.6
[PASS] Pandas: 3.1.0
[PASS] Pillow: 13.0.0
[PASS] OpenCV: 5.0.0.93
[PASS] Matplotlib: 3.11.2
[PASS] Pydicom: 3.0.2

KIỂM TRA THIẾT BỊ
PyTorch: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

-------------------------------------------------------
KẾT QUẢ KHỞI TẠO
Thư mục ứng dụng: C:\Breast-Cancer-CBIS-DDSM\WebDemo
File cấu hình: C:\Breast-Cancer-CBIS-DDSM\WebDemo\config\app_config.json
Báo cáo môi trường: C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports\nb11_1_environment_report.json

Tất cả thư viện cần 

In [14]:
# 11.2 — Kiểm tra checkpoint
from pathlib import Path
import json
import torch

# 1. XÁC ĐỊNH ĐƯỜNG DẪN CHECKPOINT

YOLO_FINETUNE_RUN = (
    TRAINING_ROOT
    / "nb08_7_3_yolov5_640_finetuning"
    / "yolov5n_640_finetune_lr001_20261116_205312"
)

YOLO_BASELINE_RUN = (
    TRAINING_ROOT
    / "nb08_6_yolov5_baseline"
    / "yolov5n_baseline_640_b4_20261115_013121"
)

YOLO_FINETUNE_CHECKPOINT = (
    YOLO_FINETUNE_RUN / "weights" / "best.pt"
)

YOLO_BASELINE_CHECKPOINT = (
    YOLO_BASELINE_RUN / "weights" / "best.pt"
)

INCEPTION_CHECKPOINT = (
    TRAINING_ROOT
    / "nb08_8_cnn_training"
    / "nb08_8_6_cnn_training"
    / "InceptionV3"
    / "best_checkpoint.pth"
)

# 2. KIỂM TRA CHECKPOINT YOLO

print("11.2 — KIỂM TRA CHECKPOINT VÀ CẤU HÌNH MÔ HÌNH")
print("-" * 60)

yolo_checkpoints = {
    "Fine-tuned YOLOv5n 640": YOLO_FINETUNE_CHECKPOINT,
    "Baseline YOLOv5n 640": YOLO_BASELINE_CHECKPOINT,
}

yolo_report = {}

for model_name, checkpoint_path in yolo_checkpoints.items():
    exists = checkpoint_path.is_file()

    info = {
        "path": str(checkpoint_path),
        "exists": exists,
        "size_mb": None,
    }

    if exists:
        info["size_mb"] = round(
            checkpoint_path.stat().st_size / (1024 ** 2), 2
        )

    yolo_report[model_name] = info

    print(f"\n{model_name}")
    print(f"Trạng thái: {'PASS' if exists else 'MISSING'}")
    print(f"Đường dẫn: {checkpoint_path}")

    if exists:
        print(f"Dung lượng: {info['size_mb']} MB")

# 3. KIỂM TRA CHECKPOINT INCEPTIONV3

print("\n" + "-" * 60)
print("INCEPTIONV3 CHECKPOINT")

inception_exists = INCEPTION_CHECKPOINT.is_file()

inception_report = {
    "path": str(INCEPTION_CHECKPOINT),
    "exists": inception_exists,
    "size_mb": None,
    "checkpoint_keys": [],
    "state_dict_candidates": [],
    "tensor_count": None,
    "load_status": "NOT_CHECKED",
}

print(f"Trạng thái: {'PASS' if inception_exists else 'MISSING'}")
print(f"Đường dẫn: {INCEPTION_CHECKPOINT}")

if inception_exists:
    inception_report["size_mb"] = round(
        INCEPTION_CHECKPOINT.stat().st_size / (1024 ** 2), 2
    )

    print(f"Dung lượng: {inception_report['size_mb']} MB")

    try:
        checkpoint = torch.load(
            INCEPTION_CHECKPOINT,
            map_location="cpu",
            weights_only=True,
        )

        inception_report["load_status"] = "PASS"

        if isinstance(checkpoint, dict):
            inception_report["checkpoint_keys"] = [
                str(key) for key in checkpoint.keys()
            ]

            print("\nCác thành phần checkpoint:")

            for key, value in checkpoint.items():
                print(f"- {key}: {type(value).__name__}")

                if isinstance(value, dict) and value:
                    tensor_items = [
                        item for item in value.values()
                        if isinstance(item, torch.Tensor)
                    ]

                    if tensor_items:
                        inception_report[
                            "state_dict_candidates"
                        ].append(str(key))

                        print(
                            f"  Số tensor: {len(tensor_items)}"
                        )

                        if inception_report["tensor_count"] is None:
                            inception_report["tensor_count"] = len(
                                tensor_items
                            )

        else:
            print(
                "Cấu trúc checkpoint:",
                type(checkpoint).__name__,
            )

    except Exception as error:
        inception_report["load_status"] = "ERROR"
        inception_report["load_error"] = str(error)

        print("[WARNING] Không đọc được cấu trúc checkpoint.")
        print(f"Chi tiết: {error}")

else:
    print("[WARNING] Chưa tìm thấy checkpoint InceptionV3.")

# 4. CẤU HÌNH MÔ HÌNH SỬ DỤNG TRONG WEB DEMO

model_config = {
    "architecture": "Two-stage Pipeline",

    "detection": {
        "name": "Fine-tuned YOLOv5n 640",
        "checkpoint": str(YOLO_FINETUNE_CHECKPOINT),
        "image_size": 640,
        "confidence_threshold": 0.20,
        "nms_iou_threshold": 0.60,
    },

    "classification": {
        "name": "InceptionV3",
        "checkpoint": str(INCEPTION_CHECKPOINT),
        "image_size": 299,
        "classification_threshold": 0.30,
        "transform_input": True,
        "crop_margin": 0.10,
        "classes": ["BENIGN", "MALIGNANT"],
    },

    "input": {
        "formats": ["jpg", "jpeg", "png", "dcm"],
        "max_upload_size_mb": 20,
        "max_images_per_run": 1,
    },
}

# 5. XÁC NHẬN CÁC ĐIỀU KIỆN KIỂM TRA

checks = {
    "Fine-tuned YOLO checkpoint": (
        YOLO_FINETUNE_CHECKPOINT.is_file()
    ),
    "InceptionV3 checkpoint": inception_exists,
    "InceptionV3 checkpoint readable": (
        inception_report["load_status"] == "PASS"
    ),
    "YOLOv5 repository": YOLO_REPO.is_dir(),
}

print("\n" + "-" * 60)
print("TỔNG HỢP KIỂM TRA")

for check_name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {check_name}")

# 6. LƯU CẤU HÌNH MÔ HÌNH

model_config_path = CONFIG_ROOT / "model_config.json"

with open(model_config_path, "w", encoding="utf-8") as file:
    json.dump(
        model_config,
        file,
        ensure_ascii=False,
        indent=4,
    )

# 7. LƯU BÁO CÁO

checkpoint_report = {
    "yolo_checkpoints": yolo_report,
    "inception_checkpoint": inception_report,
    "checks": checks,
    "model_config": model_config,
}

checkpoint_report_path = (
    REPORT_ROOT / "nb11_2_checkpoint_report.json"
)

with open(
    checkpoint_report_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        checkpoint_report,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("\nFILE ĐÃ LƯU")
print(f"Model config: {model_config_path}")
print(f"Checkpoint report: {checkpoint_report_path}")

print("\nTRẠNG THÁI 11.2")

if all(checks.values()):
    print("PASS — Các checkpoint và cấu hình cơ bản đã được xác nhận.")
else:
    print("NEED REVIEW — Có thành phần chưa đạt yêu cầu.")
    print("Chưa chuyển sang tích hợp mô hình.")

print("\nHOÀN TẤT CELL 11.2")


11.2 — KIỂM TRA CHECKPOINT VÀ CẤU HÌNH MÔ HÌNH
------------------------------------------------------------

Fine-tuned YOLOv5n 640
Trạng thái: PASS
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_7_3_yolov5_640_finetuning\yolov5n_640_finetune_lr001_20261116_205312\weights\best.pt
Dung lượng: 3.61 MB

Baseline YOLOv5n 640
Trạng thái: PASS
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_6_yolov5_baseline\yolov5n_baseline_640_b4_20261115_013121\weights\best.pt
Dung lượng: 3.61 MB

------------------------------------------------------------
INCEPTIONV3 CHECKPOINT
Trạng thái: PASS
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_8_cnn_training\nb08_8_6_cnn_training\InceptionV3\best_checkpoint.pth
Dung lượng: 93.22 MB

Các thành phần checkpoint:
- model_name: str
- epoch: int
- model_state_dict: OrderedDict
  Số tensor: 580
- best_validation_loss: float
- validation_metrics: dict
- image_size: int
- cl

In [15]:
# 11.3 — Tạo module image_preprocessing.py

import importlib.util

# 1. XÁC ĐỊNH ĐƯỜNG DẪN MODULE

PREPROCESSING_FILE = (
    UTILS_ROOT / "image_preprocessing.py"
)

# 2. NỘI DUNG MODULE TIỀN XỬ LÝ ẢNH

module_code = r'''
from pathlib import Path
from io import BytesIO
import numpy as np
from PIL import Image, ImageOps
import pydicom


SUPPORTED_FORMATS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".dcm",
}

MAX_UPLOAD_SIZE_MB = 20


class ImageProcessingError(Exception):
    """Lỗi trong quá trình đọc và tiền xử lý ảnh."""


def normalize_percentile(image_array, p_low=1, p_high=99):
    """
    Chuẩn hóa ảnh theo percentile P1-P99.

    Trả về ảnh grayscale uint8 trong khoảng 0-255.
    """

    image_array = np.asarray(image_array)

    if image_array.ndim != 2:
        raise ImageProcessingError(
            "Ảnh đầu vào phải là ảnh xám hai chiều."
        )

    if not np.isfinite(image_array).all():
        raise ImageProcessingError(
            "Ảnh chứa giá trị pixel không hợp lệ."
        )

    image_array = image_array.astype(np.float32)

    lower = float(np.percentile(image_array, p_low))
    upper = float(np.percentile(image_array, p_high))

    if upper <= lower:
        raise ImageProcessingError(
            "Không thể chuẩn hóa ảnh do khoảng cường độ pixel không hợp lệ."
        )

    normalized = np.clip(
        (image_array - lower) / (upper - lower),
        0,
        1,
    )

    normalized = np.round(normalized * 255).astype(np.uint8)

    return normalized


def read_source_bytes(source):
    """
    Đọc dữ liệu từ đường dẫn, bytes hoặc file-like object.
    """

    if isinstance(source, bytes):
        return source

    if isinstance(source, bytearray):
        return bytes(source)

    if isinstance(source, (str, Path)):
        path = Path(source)

        if not path.is_file():
            raise ImageProcessingError(
                f"Không tìm thấy file: {path}"
            )

        return path.read_bytes()

    if hasattr(source, "getvalue"):
        return source.getvalue()

    if hasattr(source, "read"):
        if hasattr(source, "seek"):
            source.seek(0)

        return source.read()

    raise ImageProcessingError(
        "Kiểu dữ liệu đầu vào không được hỗ trợ."
    )


def get_image_metadata(
    image_array,
    filename,
    file_size,
    image_format,
    dicom_metadata=None,
):
    """
    Tạo metadata cơ bản của ảnh.
    """

    height, width = image_array.shape

    return {
        "filename": filename,
        "format": image_format,
        "width": int(width),
        "height": int(height),
        "dimensions": f"{width} x {height}",
        "file_size_bytes": int(file_size),
        "file_size_mb": round(
            file_size / (1024 ** 2), 3
        ),
        "color_mode": "Grayscale",
        "dicom": dicom_metadata or {},
    }


def load_raster_image(image_bytes):
    """
    Đọc ảnh JPG, JPEG hoặc PNG.
    """

    try:
        with Image.open(BytesIO(image_bytes)) as image:
            image = ImageOps.exif_transpose(image)
            image_gray = np.asarray(
                image.convert("L")
            )

    except Exception as error:
        raise ImageProcessingError(
            f"Không thể đọc ảnh raster: {error}"
        )

    if image_gray.ndim != 2:
        raise ImageProcessingError(
            "Ảnh raster không có cấu trúc grayscale hợp lệ."
        )

    return image_gray


def load_dicom_image(image_bytes):
    """
    Đọc ảnh DICOM và lấy dữ liệu pixel.

    CBIS-DDSM sử dụng ảnh mammogram grayscale.
    Quy trình chuẩn hóa chính sử dụng P1-P99.
    """

    try:
        dataset = pydicom.dcmread(
            BytesIO(image_bytes)
        )

        image_array = dataset.pixel_array

    except Exception as error:
        raise ImageProcessingError(
            f"Không thể đọc dữ liệu DICOM: {error}"
        )

    image_array = np.asarray(image_array)

    if image_array.ndim == 3 and image_array.shape[0] == 1:
        image_array = image_array[0]

    if image_array.ndim != 2:
        raise ImageProcessingError(
            "DICOM không phải ảnh grayscale hai chiều "
            "hoặc chứa nhiều frame không được hỗ trợ."
        )

    dicom_metadata = {
        "modality": str(
            getattr(dataset, "Modality", "")
        ),
        "photometric_interpretation": str(
            getattr(dataset, "PhotometricInterpretation", "")
        ),
        "rows": int(
            getattr(dataset, "Rows", image_array.shape[0])
        ),
        "columns": int(
            getattr(dataset, "Columns", image_array.shape[1])
        ),
        "manufacturer": str(
            getattr(dataset, "Manufacturer", "")
        ),
    }

    return image_array, dicom_metadata


def load_medical_image(
    source,
    filename=None,
    max_size_mb=MAX_UPLOAD_SIZE_MB,
):
    """
    Đọc và tiền xử lý ảnh đầu vào.

    Kết quả:
    - image_gray: ảnh grayscale uint8.
    - image_rgb: ảnh RGB uint8.
    - metadata: thông tin ảnh.
    - image_array_raw: dữ liệu pixel trước chuẩn hóa.
    """

    if filename is None:
        if isinstance(source, (str, Path)):
            filename = Path(source).name
        else:
            filename = "uploaded_image"

    filename = str(filename)

    extension = Path(filename).suffix.lower()

    if extension not in SUPPORTED_FORMATS:
        raise ImageProcessingError(
            "Định dạng không được hỗ trợ. "
            "Chỉ chấp nhận JPG, JPEG, PNG và DICOM."
        )

    image_bytes = read_source_bytes(source)

    if not image_bytes:
        raise ImageProcessingError(
            "File ảnh rỗng."
        )

    file_size = len(image_bytes)
    max_size_bytes = int(max_size_mb * 1024 * 1024)

    if file_size > max_size_bytes:
        raise ImageProcessingError(
            f"Dung lượng ảnh vượt quá {max_size_mb} MB."
        )

    dicom_metadata = {}

    if extension == ".dcm":
        image_array_raw, dicom_metadata = (
            load_dicom_image(image_bytes)
        )

        image_gray = normalize_percentile(
            image_array_raw,
            p_low=1,
            p_high=99,
        )

        photometric = dicom_metadata[
            "photometric_interpretation"
        ].upper()

        if photometric == "MONOCHROME1":
            image_gray = 255 - image_gray

        elif photometric not in ("", "MONOCHROME2"):
            raise ImageProcessingError(
                "Photometric Interpretation của DICOM "
                "không được hỗ trợ."
            )

        image_format = "DICOM"

    else:
        image_array_raw = load_raster_image(
            image_bytes
        )

        image_gray = normalize_percentile(
            image_array_raw,
            p_low=1,
            p_high=99,
        )

        image_format = extension.replace(".", "").upper()

    if image_gray.size == 0:
        raise ImageProcessingError(
            "Ảnh sau tiền xử lý không có dữ liệu."
        )

    image_rgb = np.repeat(
        image_gray[:, :, np.newaxis],
        3,
        axis=2,
    )

    metadata = get_image_metadata(
        image_gray,
        filename,
        file_size,
        image_format,
        dicom_metadata,
    )

    return {
        "image_gray": image_gray,
        "image_rgb": image_rgb,
        "image_array_raw": image_array_raw,
        "metadata": metadata,
    }
'''

# 3. GHI MODULE RA FILE

PREPROCESSING_FILE.write_text(
    module_code,
    encoding="utf-8",
)

print("11.3 — TẠO MODULE TIỀN XỬ LÝ ẢNH")
print("-" * 55)
print(f"[PASS] Đã tạo: {PREPROCESSING_FILE}")

# 4. NẠP MODULE ĐỂ KIỂM TRA

spec = importlib.util.spec_from_file_location(
    "image_preprocessing",
    PREPROCESSING_FILE,
)

image_preprocessing = importlib.util.module_from_spec(spec)

sys.modules[spec.name] = image_preprocessing
spec.loader.exec_module(image_preprocessing)

print("[PASS] Import module thành công.")

# 5. KIỂM TRA CHỨC NĂNG CHUẨN HÓA

import numpy as np

test_array = np.arange(
    10000,
    dtype=np.uint16,
).reshape(100, 100)

normalized_test = (
    image_preprocessing.normalize_percentile(
        test_array,
        p_low=1,
        p_high=99,
    )
)

assert normalized_test.dtype == np.uint8
assert normalized_test.shape == (100, 100)
assert normalized_test.min() == 0
assert normalized_test.max() == 255

print("[PASS] Kiểm tra chuẩn hóa P1-P99.")
print(f"  Kiểu dữ liệu: {normalized_test.dtype}")
print(f"  Kích thước: {normalized_test.shape}")
print(
    f"  Pixel range: "
    f"{normalized_test.min()} - {normalized_test.max()}"
)

# 6. KIỂM TRA CHUYỂN ẢNH RGB

test_rgb = np.repeat(
    normalized_test[:, :, np.newaxis],
    3,
    axis=2,
)

assert test_rgb.shape == (100, 100, 3)
assert test_rgb.dtype == np.uint8

print("[PASS] Kiểm tra chuyển đổi Grayscale sang RGB.")

# 7. KIỂM TRA ĐỊNH DẠNG VÀ DUNG LƯỢNG

assert image_preprocessing.SUPPORTED_FORMATS == {
    ".jpg", ".jpeg", ".png", ".dcm"
}

assert image_preprocessing.MAX_UPLOAD_SIZE_MB == 20

print("[PASS] Kiểm tra danh sách định dạng.")
print("[PASS] Giới hạn dung lượng mặc định: 20 MB.")

# 8. LƯU BÁO CÁO

preprocessing_report = {
    "module": str(PREPROCESSING_FILE),
    "module_created": PREPROCESSING_FILE.is_file(),
    "supported_formats": sorted(
        image_preprocessing.SUPPORTED_FORMATS
    ),
    "max_upload_size_mb": (
        image_preprocessing.MAX_UPLOAD_SIZE_MB
    ),
    "normalization": "Percentile P1-P99",
    "output_grayscale": "uint8",
    "output_rgb": "uint8, 3 channels",
    "dicom_support": True,
    "monochrome1_inversion": True,
    "tests": {
        "module_import": True,
        "normalization": True,
        "rgb_conversion": True,
        "format_check": True,
        "size_limit_check": True,
    },
}

preprocessing_report_path = (
    REPORT_ROOT / "nb11_3_preprocessing_report.json"
)

with open(
    preprocessing_report_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preprocessing_report,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("\n" + "-" * 55)
print("TỔNG KẾT 11.3")
print("Module tiền xử lý ảnh: PASS")
print(f"Báo cáo: {preprocessing_report_path}")
print("HOÀN TẤT CELL 11.3")


11.3 — TẠO MODULE TIỀN XỬ LÝ ẢNH
-------------------------------------------------------
[PASS] Đã tạo: C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils\image_preprocessing.py
[PASS] Import module thành công.
[PASS] Kiểm tra chuẩn hóa P1-P99.
  Kiểu dữ liệu: uint8
  Kích thước: (100, 100)
  Pixel range: 0 - 255
[PASS] Kiểm tra chuyển đổi Grayscale sang RGB.
[PASS] Kiểm tra danh sách định dạng.
[PASS] Giới hạn dung lượng mặc định: 20 MB.

-------------------------------------------------------
TỔNG KẾT 11.3
Module tiền xử lý ảnh: PASS
Báo cáo: C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports\nb11_3_preprocessing_report.json
HOÀN TẤT CELL 11.3


In [16]:
# 11.4 — TÍCH HỢP YOLOv5n

import traceback
import numpy as np
import torch

# 1. Khai báo đường dẫn và cấu hình
PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
WEBDEMO_ROOT = PROJECT_ROOT / "WebDemo"
APP_ROOT = WEBDEMO_ROOT / "app"
UTILS_DIR = APP_ROOT / "utils"
CONFIG_DIR = WEBDEMO_ROOT / "config"
REPORT_DIR = WEBDEMO_ROOT / "reports"

YOLO_REPO = PROJECT_ROOT / "external" / "yolov5"

YOLO_WEIGHTS = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_7_3_yolov5_640_finetuning"
    / "yolov5n_640_finetune_lr001_20261116_205312"
    / "weights" / "best.pt"
)

YOLO_CONF_THRESHOLD = 0.20
YOLO_NMS_IOU = 0.60
YOLO_IMAGE_SIZE = 640
YOLO_MAX_DETECTIONS = 300

MODULE_PATH = UTILS_DIR / "yolo_detector.py"
CONFIG_PATH = CONFIG_DIR / "model_config.json"
REPORT_PATH = REPORT_DIR / "nb11_4_yolo_report.json"

# 2. Kiểm tra các thành phần đầu vào
required_paths = {
    "Thư mục dự án": PROJECT_ROOT,
    "YOLOv5 repository": YOLO_REPO,
    "Checkpoint Fine-tuned": YOLO_WEIGHTS,
    "Thư mục tiện ích": UTILS_DIR,
    "Thư mục báo cáo": REPORT_DIR,
}

path_status = {}
for name, path in required_paths.items():
    exists = path.exists()
    path_status[name] = exists
    print(f"[{'PASS' if exists else 'FAIL'}] {name}: {path}")

if not all(path_status.values()):
    raise FileNotFoundError(
        "Thiếu thành phần đầu vào. Kiểm tra các đường dẫn FAIL ở trên."
    )

# 3. Tạo module YOLOv5n
module_code = r'''
from pathlib import Path
import sys
import numpy as np
import torch
import cv2

# Bổ sung tương thích NumPy cho YOLOv5 trong môi trường hiện tại
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid

YOLO_REPO = Path(r"C:\Breast-Cancer-CBIS-DDSM\external\yolov5")

if str(YOLO_REPO) not in sys.path:
    sys.path.insert(0, str(YOLO_REPO))

from models.common import DetectMultiBackend
from utils.augmentations import letterbox
from utils.general import non_max_suppression, scale_boxes


class YOLOMassDetector:
    """Phát hiện và định vị Mass bằng YOLOv5n Fine-tuned."""

    def __init__(
        self,
        weights,
        conf_threshold=0.20,
        nms_iou=0.60,
        image_size=640,
        max_detections=300,
        device=None
    ):
        self.weights = Path(weights)
        self.conf_threshold = float(conf_threshold)
        self.nms_iou = float(nms_iou)
        self.image_size = int(image_size)
        self.max_detections = int(max_detections)

        if not self.weights.is_file():
            raise FileNotFoundError(
                f"Không tìm thấy checkpoint: {self.weights}"
            )

        if device is None:
            device = "cuda:0" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device)

        self.model = DetectMultiBackend(
            str(self.weights),
            device=self.device,
            dnn=False,
            data=None,
            fp16=False
        )

        self.model.eval()
        self.stride = int(self.model.stride)
        self.names = self.model.names

        self.model.warmup(
            imgsz=(1, 3, self.image_size, self.image_size)
        )

    def detect(self, image_rgb):
        """
        Đầu vào:
            image_rgb: ảnh RGB uint8, dạng H x W x 3.

        Đầu ra:
            Dictionary gồm ảnh gốc, danh sách bounding box,
            số lượng Mass và thông tin kích thước.
        """
        image_rgb = np.asarray(image_rgb)

        if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
            raise ValueError("Ảnh đầu vào phải có dạng H x W x 3 (RGB).")

        if image_rgb.dtype != np.uint8:
            raise TypeError("Ảnh đầu vào phải có kiểu dữ liệu uint8.")

        if image_rgb.size == 0:
            raise ValueError("Ảnh đầu vào rỗng.")

        original_height, original_width = image_rgb.shape[:2]

        # Letterbox giữ tỷ lệ ảnh và bổ sung vùng đệm
        image_input = letterbox(
            image_rgb,
            new_shape=self.image_size,
            stride=self.stride,
            auto=False
        )[0]

        # Ảnh đầu vào đã là RGB nên không đảo thứ tự kênh
        image_input = image_input.transpose((2, 0, 1))
        image_input = np.ascontiguousarray(image_input)

        tensor = torch.from_numpy(image_input).to(self.device)
        tensor = tensor.float() / 255.0
        tensor = tensor.unsqueeze(0)

        with torch.inference_mode():
            prediction = self.model(tensor)
            detections = non_max_suppression(
                prediction,
                conf_thres=self.conf_threshold,
                iou_thres=self.nms_iou,
                classes=None,
                agnostic=False,
                max_det=self.max_detections
            )

        boxes = []

        if len(detections) > 0 and len(detections[0]) > 0:
            det = detections[0].clone()

            # Đưa tọa độ từ ảnh letterbox về ảnh gốc
            det[:, :4] = scale_boxes(
                tensor.shape[2:],
                det[:, :4],
                image_rgb.shape
            ).round()

            for row in det.cpu().numpy():
                x1, y1, x2, y2, confidence, class_id = row

                x1 = int(np.clip(x1, 0, original_width - 1))
                y1 = int(np.clip(y1, 0, original_height - 1))
                x2 = int(np.clip(x2, 0, original_width - 1))
                y2 = int(np.clip(y2, 0, original_height - 1))

                if x2 <= x1 or y2 <= y1:
                    continue

                class_id = int(class_id)

                if isinstance(self.names, dict):
                    class_name = self.names.get(class_id, str(class_id))
                else:
                    class_name = (
                        self.names[class_id]
                        if class_id < len(self.names)
                        else str(class_id)
                    )

                boxes.append({
                    "box_id": len(boxes) + 1,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "confidence": float(confidence),
                    "class_id": class_id,
                    "class_name": str(class_name)
                })

        return {
            "boxes": boxes,
            "num_detections": len(boxes),
            "image_width": original_width,
            "image_height": original_height,
            "conf_threshold": self.conf_threshold,
            "nms_iou": self.nms_iou,
            "image_size": self.image_size
        }

    @staticmethod
    def draw_detections(image_rgb, detections):
        """Vẽ bounding box và mã vùng phát hiện lên ảnh RGB."""
        image_bgr = cv2.cvtColor(
            np.asarray(image_rgb).copy(),
            cv2.COLOR_RGB2BGR
        )

        for box in detections["boxes"]:
            x1, y1 = box["x1"], box["y1"]
            x2, y2 = box["x2"], box["y2"]

            confidence = box["confidence"]
            box_id = box["box_id"]

            cv2.rectangle(
                image_bgr,
                (x1, y1),
                (x2, y2),
                (180, 45, 120),
                2
            )

            label = f"Mass {box_id} | {confidence:.2f}"

            text_y = max(y1 - 8, 20)

            cv2.putText(
                image_bgr,
                label,
                (x1, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (180, 45, 120),
                2,
                cv2.LINE_AA
            )

        return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
'''

MODULE_PATH.write_text(module_code, encoding="utf-8")
print(f"\n[PASS] Đã tạo module: {MODULE_PATH}")

# 4. Import module
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

spec = importlib.util.spec_from_file_location(
    "yolo_detector",
    MODULE_PATH
)
yolo_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(yolo_module)

YOLOMassDetector = yolo_module.YOLOMassDetector

print("[PASS] Import module YOLOMassDetector thành công.")

# 5. Nạp checkpoint Fine-tuned
detector = YOLOMassDetector(
    weights=YOLO_WEIGHTS,
    conf_threshold=YOLO_CONF_THRESHOLD,
    nms_iou=YOLO_NMS_IOU,
    image_size=YOLO_IMAGE_SIZE,
    max_detections=YOLO_MAX_DETECTIONS
)

print("[PASS] Nạp checkpoint Fine-tuned YOLOv5n thành công.")
print(f"  Device: {detector.device}")
print(f"  Stride: {detector.stride}")
print(f"  Class names: {detector.names}")

if detector.device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(detector.device)}")

# 6. Kiểm tra suy luận bằng ảnh giả lập
# Ảnh đen chỉ dùng để kiểm tra luồng xử lý, không dùng đánh giá mô hình.
test_image = np.zeros((640, 640, 3), dtype=np.uint8)

test_result = detector.detect(test_image)
test_annotated = detector.draw_detections(
    test_image,
    test_result
)

assert test_annotated.shape == test_image.shape
assert test_annotated.dtype == np.uint8
assert isinstance(test_result["boxes"], list)

print("[PASS] Suy luận thử hoàn tất.")
print(f"  Kích thước ảnh: {test_image.shape}")
print(f"  Số vùng phát hiện trên ảnh giả lập: {test_result['num_detections']}")
print(f"  Kích thước ảnh kết quả: {test_annotated.shape}")

# 7. Cập nhật cấu hình mô hình
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        model_config = json.load(file)
else:
    model_config = {}

model_config["yolo_detector"] = {
    "model_name": "YOLOv5n Fine-tuned",
    "weights": str(YOLO_WEIGHTS),
    "image_size": YOLO_IMAGE_SIZE,
    "confidence_threshold": YOLO_CONF_THRESHOLD,
    "nms_iou_threshold": YOLO_NMS_IOU,
    "max_detections": YOLO_MAX_DETECTIONS,
    "class_names": detector.names,
    "device": str(detector.device)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        model_config,
        file,
        ensure_ascii=False,
        indent=4,
        default=str
    )

print(f"[PASS] Đã cập nhật cấu hình: {CONFIG_PATH}")

# 8. Lưu báo cáo kiểm tra
report = {
    "notebook_section": "11.4",
    "status": "PASS",
    "module_path": str(MODULE_PATH),
    "checkpoint": str(YOLO_WEIGHTS),
    "checkpoint_exists": YOLO_WEIGHTS.is_file(),
    "device": str(detector.device),
    "class_names": detector.names,
    "image_size": YOLO_IMAGE_SIZE,
    "confidence_threshold": YOLO_CONF_THRESHOLD,
    "nms_iou_threshold": YOLO_NMS_IOU,
    "max_detections": YOLO_MAX_DETECTIONS,
    "synthetic_inference": {
        "status": "PASS",
        "image_shape": list(test_image.shape),
        "image_dtype": str(test_image.dtype),
        "num_detections": test_result["num_detections"],
        "annotated_image_shape": list(test_annotated.shape)
    },
    "note": (
        "Ảnh giả lập chỉ kiểm tra luồng suy luận kỹ thuật, "
        "không đại diện cho hiệu quả phát hiện trên ảnh nhũ ảnh."
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=4, default=str)

print(f"\n[PASS] Báo cáo: {REPORT_PATH}")
print("\nTỔNG KẾT 11.4")
print("Module YOLOv5n: PASS")
print("Nạp checkpoint: PASS")
print("Kiểm tra suy luận kỹ thuật: PASS")
print("HOÀN TẤT CELL 11.4")


[PASS] Thư mục dự án: C:\Breast-Cancer-CBIS-DDSM
[PASS] YOLOv5 repository: C:\Breast-Cancer-CBIS-DDSM\external\yolov5
[PASS] Checkpoint Fine-tuned: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_7_3_yolov5_640_finetuning\yolov5n_640_finetune_lr001_20261116_205312\weights\best.pt
[PASS] Thư mục tiện ích: C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils
[PASS] Thư mục báo cáo: C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports

[PASS] Đã tạo module: C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils\yolo_detector.py
[PASS] Import module YOLOMassDetector thành công.


C:\Breast-Cancer-CBIS-DDSM\external\yolov5\models\experimental.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(attempt_download(w), map_location='cpu'

[PASS] Nạp checkpoint Fine-tuned YOLOv5n thành công.
  Device: cuda:0
  Stride: 32
  Class names: {0: 'mass'}
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU
[PASS] Suy luận thử hoàn tất.
  Kích thước ảnh: (640, 640, 3)
  Số vùng phát hiện trên ảnh giả lập: 0
  Kích thước ảnh kết quả: (640, 640, 3)
[PASS] Đã cập nhật cấu hình: C:\Breast-Cancer-CBIS-DDSM\WebDemo\config\model_config.json

[PASS] Báo cáo: C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports\nb11_4_yolo_report.json

TỔNG KẾT 11.4
Module YOLOv5n: PASS
Nạp checkpoint: PASS
Kiểm tra suy luận kỹ thuật: PASS
HOÀN TẤT CELL 11.4


In [17]:
# 11.5 — TÍCH HỢP InceptionV3

import torch
import torch.nn as nn
import torchvision
from torchvision.models import inception_v3
from PIL import Image

# 1. Khai báo đường dẫn và cấu hình
PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
WEBDEMO_ROOT = PROJECT_ROOT / "WebDemo"
APP_ROOT = WEBDEMO_ROOT / "app"
UTILS_DIR = APP_ROOT / "utils"
CONFIG_DIR = WEBDEMO_ROOT / "config"
REPORT_DIR = WEBDEMO_ROOT / "reports"

CNN_CHECKPOINT = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_8_cnn_training"
    / "nb08_8_6_cnn_training"
    / "InceptionV3" / "best_checkpoint.pth"
)

MODULE_PATH = UTILS_DIR / "inception_classifier.py"
CONFIG_PATH = CONFIG_DIR / "model_config.json"
REPORT_PATH = REPORT_DIR / "nb11_5_inception_report.json"

IMAGE_SIZE = 299
CLASSIFICATION_THRESHOLD = 0.30
CLASS_NAMES = ["BENIGN", "MALIGNANT"]

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

# 2. Kiểm tra đầu vào
required_paths = {
    "Thư mục dự án": PROJECT_ROOT,
    "Checkpoint InceptionV3": CNN_CHECKPOINT,
    "Thư mục tiện ích": UTILS_DIR,
    "Thư mục cấu hình": CONFIG_DIR,
    "Thư mục báo cáo": REPORT_DIR,
}

path_status = {}
for name, path in required_paths.items():
    exists = path.exists()
    path_status[name] = exists
    print(f"[{'PASS' if exists else 'FAIL'}] {name}: {path}")

if not all(path_status.values()):
    raise FileNotFoundError(
        "Thiếu thành phần đầu vào. Kiểm tra các đường dẫn FAIL."
    )

# 3. Đọc checkpoint và kiểm tra cấu trúc
checkpoint = torch.load(
    CNN_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

if not isinstance(checkpoint, dict):
    raise TypeError("Checkpoint không có cấu trúc Dictionary.")

if "model_state_dict" not in checkpoint:
    raise KeyError("Checkpoint không chứa model_state_dict.")

state_dict = checkpoint["model_state_dict"]

if not isinstance(state_dict, dict):
    raise TypeError("model_state_dict không hợp lệ.")

# Loại bỏ tiền tố DataParallel nếu có
clean_state_dict = {}
for key, value in state_dict.items():
    clean_key = key.replace("module.", "", 1) if key.startswith("module.") else key
    clean_state_dict[clean_key] = value

has_aux_logits = any(
    key.startswith("AuxLogits.")
    for key in clean_state_dict
)

if "fc.weight" not in clean_state_dict:
    raise KeyError("Không tìm thấy lớp fc.weight trong checkpoint.")

num_classes = int(clean_state_dict["fc.weight"].shape[0])

if num_classes != 2:
    raise ValueError(
        f"Checkpoint có {num_classes} đầu ra, yêu cầu mô hình nhị phân."
    )

if tuple(clean_state_dict["fc.weight"].shape) != (2, 2048):
    raise ValueError(
        "Kích thước lớp phân loại chính không đúng với InceptionV3 2 lớp."
    )

print("\n[PASS] Đọc checkpoint bằng weights_only=True.")
print(f"  Model name: {checkpoint.get('model_name', 'Không có thông tin')}")
print(f"  Epoch tốt nhất: {checkpoint.get('epoch', 'Không có thông tin')}")
print(f"  Số tensor: {len(clean_state_dict)}")
print(f"  AuxLogits: {'Có' if has_aux_logits else 'Không'}")
print(f"  Số lớp đầu ra: {num_classes}")

# 4. Tạo module InceptionV3
module_code = r'''
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import inception_v3
from PIL import Image


class InceptionMassClassifier:
    """Phân loại bệnh lý Mass bằng InceptionV3 đã huấn luyện."""

    CLASS_NAMES = ["BENIGN", "MALIGNANT"]

    def __init__(
        self,
        checkpoint_path,
        image_size=299,
        threshold=0.30,
        device=None
    ):
        self.checkpoint_path = Path(checkpoint_path)
        self.image_size = int(image_size)
        self.threshold = float(threshold)

        if not self.checkpoint_path.is_file():
            raise FileNotFoundError(
                f"Không tìm thấy checkpoint: {self.checkpoint_path}"
            )

        if device is None:
            device = "cuda:0" if torch.cuda.is_available() else "cpu"

        self.device = torch.device(device)

        checkpoint = torch.load(
            self.checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        if "model_state_dict" not in checkpoint:
            raise KeyError("Checkpoint không chứa model_state_dict.")

        state_dict = checkpoint["model_state_dict"]

        # Hỗ trợ checkpoint được lưu từ DataParallel
        clean_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith("module."):
                key = key[len("module."):]
            clean_state_dict[key] = value

        has_aux_logits = any(
            key.startswith("AuxLogits.")
            for key in clean_state_dict
        )

        # Khởi tạo kiến trúc không tải lại trọng số ImageNet
        self.model = inception_v3(
            weights=None,
            aux_logits=has_aux_logits,
            transform_input=True,
            init_weights=False
        )

        # Điều chỉnh lớp phân loại chính
        self.model.fc = nn.Linear(2048, 2)

        # Điều chỉnh lớp phụ nếu checkpoint có AuxLogits
        if has_aux_logits:
            if self.model.AuxLogits is None:
                raise RuntimeError("Không khởi tạo được AuxLogits.")

            self.model.AuxLogits.fc = nn.Linear(768, 2)

        # Nạp toàn bộ trọng số, không bỏ qua tensor không khớp
        self.model.load_state_dict(
            clean_state_dict,
            strict=True
        )

        self.model.to(self.device)
        self.model.eval()

        self.checkpoint_epoch = checkpoint.get("epoch")
        self.best_validation_loss = checkpoint.get(
            "best_validation_loss"
        )
        self.has_aux_logits = has_aux_logits

        # Chuẩn hóa ImageNet
        self.mean = torch.tensor(
            [0.485, 0.456, 0.406],
            dtype=torch.float32
        ).view(3, 1, 1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225],
            dtype=torch.float32
        ).view(3, 1, 1)

    def preprocess(self, image):
        """
        Tiền xử lý ảnh RGB:
        - Chuyển về RGB.
        - Resize giữ tỷ lệ.
        - Padding đen ở giữa.
        - Chuẩn hóa ImageNet.
        - Không thay đổi tỷ lệ hình học của vùng tổn thương.
        """
        if isinstance(image, np.ndarray):
            if image.ndim != 3 or image.shape[2] != 3:
                raise ValueError("Ảnh NumPy phải có dạng H x W x 3 RGB.")

            if image.dtype != np.uint8:
                raise TypeError("Ảnh đầu vào phải có kiểu uint8.")

            pil_image = Image.fromarray(image, mode="RGB")

        elif isinstance(image, Image.Image):
            pil_image = image.convert("RGB")

        else:
            raise TypeError(
                "Đầu vào phải là ảnh PIL hoặc NumPy RGB."
            )

        original_width, original_height = pil_image.size

        if original_width <= 0 or original_height <= 0:
            raise ValueError("Kích thước ảnh không hợp lệ.")

        scale = min(
            self.image_size / original_width,
            self.image_size / original_height
        )

        resized_width = max(
            1, int(round(original_width * scale))
        )
        resized_height = max(
            1, int(round(original_height * scale))
        )

        resized_image = pil_image.resize(
            (resized_width, resized_height),
            Image.Resampling.BILINEAR
        )

        canvas = Image.new(
            "RGB",
            (self.image_size, self.image_size),
            (0, 0, 0)
        )

        left = (self.image_size - resized_width) // 2
        top = (self.image_size - resized_height) // 2

        canvas.paste(resized_image, (left, top))

        image_array = np.asarray(
            canvas,
            dtype=np.float32
        ) / 255.0

        tensor = torch.from_numpy(
            image_array.transpose(2, 0, 1).copy()
        )

        tensor = (
            tensor - self.mean
        ) / self.std

        return tensor

    def classify(self, image):
        """Trả về xác suất và nhãn phân loại của một crop."""

        input_tensor = self.preprocess(image)
        input_tensor = input_tensor.unsqueeze(0).to(self.device)

        with torch.inference_mode():
            output = self.model(input_tensor)

            # Khi ở chế độ eval, InceptionV3 thường trả logits chính.
            # Nhánh dự phòng hỗ trợ trường hợp đầu ra dạng tuple.
            if isinstance(output, (tuple, list)):
                logits = output[0]
            elif hasattr(output, "logits"):
                logits = output.logits
            else:
                logits = output

            probabilities = torch.softmax(
                logits,
                dim=1
            )[0]

        benign_probability = float(probabilities[0].item())
        malignant_probability = float(probabilities[1].item())

        predicted_label = (
            "MALIGNANT"
            if malignant_probability >= self.threshold
            else "BENIGN"
        )

        return {
            "predicted_label": predicted_label,
            "benign_probability": benign_probability,
            "malignant_probability": malignant_probability,
            "threshold": self.threshold,
            "class_probabilities": {
                "BENIGN": benign_probability,
                "MALIGNANT": malignant_probability
            }
        }
'''

MODULE_PATH.write_text(module_code, encoding="utf-8")
print(f"\n[PASS] Đã tạo module: {MODULE_PATH}")

# 5. Import module
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

spec = importlib.util.spec_from_file_location(
    "inception_classifier",
    MODULE_PATH
)

inception_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inception_module)

InceptionMassClassifier = (
    inception_module.InceptionMassClassifier
)

print("[PASS] Import module InceptionMassClassifier thành công.")

# 6. Nạp mô hình
classifier = InceptionMassClassifier(
    checkpoint_path=CNN_CHECKPOINT,
    image_size=IMAGE_SIZE,
    threshold=CLASSIFICATION_THRESHOLD,
    device=DEVICE
)

print("[PASS] Nạp checkpoint InceptionV3 thành công.")
print(f"  Device: {classifier.device}")
print(f"  Image size: {classifier.image_size}")
print(f"  Transform input: True")
print(f"  AuxLogits: {classifier.has_aux_logits}")
print(f"  Threshold: {classifier.threshold}")
print(f"  Epoch checkpoint: {classifier.checkpoint_epoch}")

if classifier.device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(classifier.device)}")

# 7. Kiểm tra tiền xử lý và suy luận bằng ảnh giả lập
# Ảnh giả lập chỉ kiểm tra luồng kỹ thuật, không đánh giá chất lượng.
test_image = np.zeros(
    (512, 384, 3),
    dtype=np.uint8
)

test_tensor = classifier.preprocess(test_image)

assert tuple(test_tensor.shape) == (3, 299, 299)
assert torch.isfinite(test_tensor).all()

test_result = classifier.classify(test_image)

assert test_result["predicted_label"] in CLASS_NAMES
assert 0.0 <= test_result["benign_probability"] <= 1.0
assert 0.0 <= test_result["malignant_probability"] <= 1.0

probability_sum = (
    test_result["benign_probability"]
    + test_result["malignant_probability"]
)

assert abs(probability_sum - 1.0) < 1e-5

print("\n[PASS] Kiểm tra tiền xử lý ảnh.")
print(f"  Kích thước tensor: {tuple(test_tensor.shape)}")
print(f"  Kiểu dữ liệu: {test_tensor.dtype}")
print(f"  Giá trị hữu hạn: {bool(torch.isfinite(test_tensor).all())}")

print("[PASS] Kiểm tra suy luận InceptionV3.")
print(f"  Nhãn dự đoán thử: {test_result['predicted_label']}")
print(f"  Xác suất BENIGN: {test_result['benign_probability']:.6f}")
print(f"  Xác suất MALIGNANT: {test_result['malignant_probability']:.6f}")
print(f"  Tổng xác suất: {probability_sum:.6f}")

# 8. Cập nhật cấu hình mô hình
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        model_config = json.load(file)
else:
    model_config = {}

model_config["inception_classifier"] = {
    "model_name": "InceptionV3",
    "checkpoint": str(CNN_CHECKPOINT),
    "image_size": IMAGE_SIZE,
    "transform_input": True,
    "resize_mode": "Aspect-preserving resize with centered black padding",
    "normalization": "ImageNet mean and standard deviation",
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "class_names": CLASS_NAMES,
    "positive_class": "MALIGNANT",
    "aux_logits": classifier.has_aux_logits,
    "device": str(classifier.device)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        model_config,
        file,
        ensure_ascii=False,
        indent=4,
        default=str
    )

print(f"\n[PASS] Đã cập nhật cấu hình: {CONFIG_PATH}")

# 9. Lưu báo cáo kiểm tra
report = {
    "notebook_section": "11.5",
    "status": "PASS",
    "module_path": str(MODULE_PATH),
    "checkpoint": str(CNN_CHECKPOINT),
    "checkpoint_exists": CNN_CHECKPOINT.is_file(),
    "checkpoint_epoch": classifier.checkpoint_epoch,
    "best_validation_loss": classifier.best_validation_loss,
    "state_dict_tensor_count": len(clean_state_dict),
    "aux_logits": classifier.has_aux_logits,
    "device": str(classifier.device),
    "image_size": IMAGE_SIZE,
    "transform_input": True,
    "normalization": "ImageNet",
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "class_names": CLASS_NAMES,
    "synthetic_test": {
        "status": "PASS",
        "input_shape": list(test_image.shape),
        "output_tensor_shape": list(test_tensor.shape),
        "predicted_label": test_result["predicted_label"],
        "benign_probability": test_result["benign_probability"],
        "malignant_probability": test_result["malignant_probability"],
        "probability_sum": probability_sum
    },
    "note": (
        "Ảnh giả lập chỉ dùng kiểm tra tiền xử lý và suy luận kỹ thuật; "
        "không đại diện cho hiệu quả phân loại trên dữ liệu Test."
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        report,
        file,
        ensure_ascii=False,
        indent=4,
        default=str
    )

print(f"[PASS] Báo cáo: {REPORT_PATH}")

print("\nTỔNG KẾT 11.5")
print("Module InceptionV3: PASS")
print("Nạp checkpoint nghiêm ngặt: PASS")
print("Tiền xử lý ảnh 299x299: PASS")
print("Suy luận và xác suất hai lớp: PASS")
print("HOÀN TẤT CELL 11.5")


[PASS] Thư mục dự án: C:\Breast-Cancer-CBIS-DDSM
[PASS] Checkpoint InceptionV3: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_8_cnn_training\nb08_8_6_cnn_training\InceptionV3\best_checkpoint.pth
[PASS] Thư mục tiện ích: C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils
[PASS] Thư mục cấu hình: C:\Breast-Cancer-CBIS-DDSM\WebDemo\config
[PASS] Thư mục báo cáo: C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports

[PASS] Đọc checkpoint bằng weights_only=True.
  Model name: InceptionV3
  Epoch tốt nhất: 8
  Số tensor: 580
  AuxLogits: Có
  Số lớp đầu ra: 2

[PASS] Đã tạo module: C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils\inception_classifier.py
[PASS] Import module InceptionMassClassifier thành công.
[PASS] Nạp checkpoint InceptionV3 thành công.
  Device: cuda:0
  Image size: 299
  Transform input: True
  AuxLogits: True
  Threshold: 0.3
  Epoch checkpoint: 8
  GPU: NVIDIA GeForce RTX 4050 Laptop GPU

[PASS] Kiểm tra tiền xử lý ảnh.
  Kích thước tensor: (3, 299, 299)
  Kiểu dữ liệu

In [38]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"

print("KIỂM TRA NGUỒN ẢNH TEST VÀ GROUND TRUTH")
print(f"Thư mục dữ liệu: {PROCESSED_DIR}")
print()

assert PROCESSED_DIR.exists(), "Không tìm thấy thư mục Data\\processed"

# ------------------------------------------------------------
# 1. Tìm các bảng CSV có khả năng chứa manifest/annotation
# ------------------------------------------------------------

keywords = [
    "test", "manifest", "annotation", "dataset",
    "detection", "mass", "split", "label", "ground_truth"
]

candidate_tables = []

for csv_path in PROCESSED_DIR.rglob("*.csv"):
    try:
        header = pd.read_csv(csv_path, nrows=0)
        columns = list(header.columns)
        name_lower = csv_path.name.lower()

        relevant_columns = [
            col for col in columns
            if any(
                key in str(col).lower()
                for key in [
                    "image", "path", "file", "dicom",
                    "bbox", "box", "label", "pathology",
                    "split", "patient", "lesion", "mask",
                    "annotation", "class"
                ]
            )
        ]

        name_relevant = any(key in name_lower for key in keywords)

        if len(relevant_columns) >= 2 or name_relevant:
            candidate_tables.append({
                "path": csv_path,
                "columns": columns,
                "relevant_columns": relevant_columns
            })

    except Exception as error:
        print(f"[SKIP] {csv_path.name}: {error}")

print(f"Tìm thấy {len(candidate_tables)} bảng CSV có khả năng liên quan.\n")

for index, item in enumerate(candidate_tables, start=1):
    print("=" * 75)
    print(f"BẢNG {index}: {item['path'].name}")
    print(f"Đường dẫn: {item['path']}")
    print(f"Cột liên quan: {item['relevant_columns']}")
    print(f"Tất cả cột: {item['columns']}")

    try:
        sample = pd.read_csv(item["path"], nrows=3)
        print("\nDỮ LIỆU MẪU:")
        display(sample)
    except Exception as error:
        print(f"Không đọc được dữ liệu mẫu: {error}")

# ------------------------------------------------------------
# 2. Tìm các thư mục có ảnh Test
# ------------------------------------------------------------

image_extensions = {
    ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".dcm"
}

print("\n\nKIỂM TRA CÁC THƯ MỤC CHỨA ẢNH")
print("-" * 75)

image_directories = []

for folder in PROCESSED_DIR.rglob("*"):
    if not folder.is_dir():
        continue

    folder_name = folder.name.lower()

    if not any(key in folder_name for key in ["test", "images", "image"]):
        continue

    try:
        count = sum(
            1 for file in folder.iterdir()
            if file.is_file() and file.suffix.lower() in image_extensions
        )

        if count > 0:
            image_directories.append((folder, count))

    except PermissionError:
        continue

for folder, count in image_directories:
    print(f"[ẢNH] {count:>6} | {folder}")

# ------------------------------------------------------------
# 3. Tóm tắt các nguồn cần đối chiếu
# ------------------------------------------------------------

print("\n\nTÓM TẮT")
print("-" * 75)
print(f"Số bảng CSV cần xem: {len(candidate_tables)}")
print(f"Số thư mục có ảnh: {len(image_directories)}")

print("\nCác bảng có tên liên quan đến Test:")
for item in candidate_tables:
    if "test" in item["path"].name.lower():
        print(item["path"])

print("\nHoàn tất kiểm tra nguồn dữ liệu.")


KIỂM TRA NGUỒN ẢNH TEST VÀ GROUND TRUTH
Thư mục dữ liệu: C:\Breast-Cancer-CBIS-DDSM\Data\processed

[SKIP] mass_image_dimension_audit.csv: No columns to parse from file
[SKIP] nb08_7_9_crop_errors.csv: No columns to parse from file
Tìm thấy 248 bảng CSV có khả năng liên quan.

BẢNG 1: dicom_index.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\dicom_index.csv
Cột liên quan: ['file_path', 'relative_path', 'PatientID']
Tất cả cột: ['file_path', 'relative_path', 'PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID', 'SeriesDescription']

DỮ LIỆU MẪU:


,file_path,relative_path,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,SeriesDescription
0,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,Calc-Test_P_00038_LEFT_MLO_1\2017-08-29-11739\...,Calc-Test_P_00038_LEFT_MLO_1,1.3.6.1.4.1.9590.100.1.2.291121996131431385353...,1.3.6.1.4.1.9590.100.1.2.188613955710170417803...,1.3.6.1.4.1.9590.100.1.2.173338537212637308823...,cropped images
1,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,Calc-Test_P_00038_LEFT_MLO_1\2017-08-29-11739\...,P_00038_LEFT_MLO_1.dcm,1.3.6.1.4.1.9590.100.1.2.291121996131431385353...,1.3.6.1.4.1.9590.100.1.2.188613955710170417803...,1.3.6.1.4.1.9590.100.1.2.408093737100776095386...,NaN
2,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,Calc-Test_P_00038_RIGHT_CC_1\2017-08-29-83105\...,P_00038_RIGHT_CC_1.dcm,1.3.6.1.4.1.9590.100.1.2.200764632211227648028...,1.3.6.1.4.1.9590.100.1.2.244876997513875090239...,1.3.6.1.4.1.9590.100.1.2.410192537111852619606...,NaN


BẢNG 2: nb02_csv_dicom_linkage.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_data_quality_linkage\nb02_csv_dicom_linkage.csv
Cột liên quan: ['split', 'path_type', 'csv_path', 'alias_filename', 'path_components', 'candidate_paths']
Tất cả cột: ['split', 'csv_row', 'path_type', 'csv_path', 'case_folder', 'study_uid', 'series_uid', 'alias_filename', 'path_components', 'candidate_count', 'candidate_paths', 'candidate_sop_uids', 'series_descriptions', 'linkage_status']

DỮ LIỆU MẪU:


,split,csv_row,path_type,csv_path,case_folder,...,candidate_count,candidate_paths,candidate_sop_uids,series_descriptions,linkage_status
0,calc_test,0,image file path,Calc-Test_P_00038_LEFT_CC/1.3.6.1.4.1.9590.100...,Calc-Test_P_00038_LEFT_CC,...,1,['P_00038_LEFT_CC.dcm\\1-63992\\9a3af421-bb04-...,['1.3.6.1.4.1.9590.100.1.2.2899237393124709664...,[],single_instance_candidate
1,calc_test,1,image file path,Calc-Test_P_00038_LEFT_MLO/1.3.6.1.4.1.9590.10...,Calc-Test_P_00038_LEFT_MLO,...,1,['P_00038_LEFT_MLO.dcm\\1-97934\\e1a1589e-55e8...,['1.3.6.1.4.1.9590.100.1.2.2656477484132627339...,[],single_instance_candidate
2,calc_test,2,image file path,Calc-Test_P_00038_RIGHT_CC/1.3.6.1.4.1.9590.10...,Calc-Test_P_00038_RIGHT_CC,...,1,['P_00038_RIGHT_CC.dcm\\1-63304\\7d487097-0a08...,['1.3.6.1.4.1.9590.100.1.2.1708498415129066103...,[],single_instance_candidate


BẢNG 3: nb02_instance_role_final.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb02_data_quality_linkage\nb02_instance_role_final.csv
Cột liên quan: ['file_path', 'path_types', 'path_types_text']
Tất cả cột: ['study_uid', 'series_uid', 'SOPInstanceUID', 'file_path', 'SeriesDescription', 'proposed_role', 'confidence_reason', 'review_status', 'pixel_unique_values', 'pixel_zero_ratio', 'pixel_shape', 'pixel_error', 'path_types', 'path_types_text', 'proposed_role_final', 'final_review_status']

DỮ LIỆU MẪU:


,study_uid,series_uid,SOPInstanceUID,file_path,SeriesDescription,...,pixel_error,path_types,path_types_text,proposed_role_final,final_review_status
0,1.3.6.1.4.1.9590.100.1.2.859354343102033567126...,1.3.6.1.4.1.9590.100.1.2.374115997511889073021...,1.3.6.1.4.1.9590.100.1.2.289923739312470966435...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,NaN,...,NaN,['image file path'],['image file path'],full_image_candidate,path_context_supported
1,1.3.6.1.4.1.9590.100.1.2.384159464510350889125...,1.3.6.1.4.1.9590.100.1.2.174390361112646747718...,1.3.6.1.4.1.9590.100.1.2.265647748413262733922...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,NaN,...,NaN,['image file path'],['image file path'],full_image_candidate,path_context_supported
2,1.3.6.1.4.1.9590.100.1.2.177706148911820252341...,1.3.6.1.4.1.9590.100.1.2.263861248711313923336...,1.3.6.1.4.1.9590.100.1.2.170849841512906610322...,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,NaN,...,NaN,['image file path'],['image file path'],full_image_candidate,path_context_supported


... (487 outputs left out)

,annotation_id,image_role,original_source_field,applied_source_field,original_csv_path,...,proposed_role_final,final_review_status,_image_path,_path_exists,_mapping_status
0,calc_train__00001,full_image,full_image_csv_path,full_image_csv_path,Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.959...,...,full_image_candidate,path_context_supported,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,True,resolved
1,calc_train__00001,cropped_image,crop_image_csv_path,crop_image_csv_path,Calc-Training_P_00005_RIGHT_CC_1/1.3.6.1.4.1.9...,...,crop_candidate,description_supported,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,True,resolved
2,calc_train__00002,full_image,full_image_csv_path,full_image_csv_path,Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.95...,...,full_image_candidate,path_context_supported,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,True,resolved


BẢNG 248: visual_roi_mapping.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\supplementary_image_role_audit_20261105_002929_370953\visual_roi_mapping.csv
Cột liên quan: ['annotation_id', '_image_path', '_path_exists', 'image_role']
Tất cả cột: ['annotation_id', '_image_path', 'study_uid', 'series_uid', 'SOPInstanceUID', 'SeriesDescription', 'expected_role', 'final_review_status', '_path_exists', 'physical_roi_status', 'image_role']

DỮ LIỆU MẪU:


,annotation_id,_image_path,study_uid,series_uid,SOPInstanceUID,...,expected_role,final_review_status,_path_exists,physical_roi_status,image_role
0,calc_train__00001,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1.3.6.1.4.1.9590.100.1.2.328778919012412769218...,1.3.6.1.4.1.9590.100.1.2.393344010211719049419...,1.3.6.1.4.1.9590.100.1.2.338899151111305765125...,...,roi_mask_candidate,description_supported,True,roi_role_match,roi_mask
1,calc_train__00002,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1.3.6.1.4.1.9590.100.1.2.675123622103196361081...,1.3.6.1.4.1.9590.100.1.2.296281207812130400303...,1.3.6.1.4.1.9590.100.1.2.874144947115837110169...,...,roi_mask_candidate,description_supported,True,roi_role_match,roi_mask
2,calc_train__00003,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1.3.6.1.4.1.9590.100.1.2.241202057913673145232...,1.3.6.1.4.1.9590.100.1.2.314135871111943890422...,1.3.6.1.4.1.9590.100.1.2.410181202110469900082...,...,roi_mask_candidate,description_supported,True,roi_role_match,roi_mask




KIỂM TRA CÁC THƯ MỤC CHỨA ẢNH
---------------------------------------------------------------------------
[ẢNH]   1696 | C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06_detection_dataset\nb06_4_8_normalized_full_images_a
[ẢNH]      1 | C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06_detection_dataset\nb06_6_image_label_spatial_synchronization
[ẢNH]    361 | C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_3_yolo_dataset\images\test
[ẢNH]      2 | C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb09_model_evaluation\nb09_3_cnn_test_evaluation
[ẢNH]      2 | C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb09_model_evaluation\nb09_5_cnn_test_comparison


TÓM TẮT
---------------------------------------------------------------------------
Số bảng CSV cần xem: 248
Số thư mục có ảnh: 5

Các bảng có tên liên quan đến Test:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb09_model_evaluation\nb09_2_yolo_test_evaluation_v2\nb09_2_yolo_test_evaluation_inventory.csv
C:\Breast-Cancer-CBIS-DDSM\D

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM\Data\processed")

SEARCH_DIRS = [
    ROOT / "nb06_detection_dataset",
    ROOT / "nb08_model_training" / "nb08_3_yolo_dataset",
    ROOT / "nb08_model_training" / "nb08_7_classification_dataset",
    ROOT / "nb08_model_training" / "nb08_8_cnn_training",
]

print("1. KIỂM TRA ẢNH TEST VÀ NHÃN YOLO")
print("-" * 75)

IMAGE_DIR = (
    ROOT / "nb08_model_training"
    / "nb08_3_yolo_dataset" / "images" / "test"
)

LABEL_DIR = (
    ROOT / "nb08_model_training"
    / "nb08_3_yolo_dataset" / "labels" / "test"
)

images = sorted([
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".tif", ".tiff"
    }
]) if IMAGE_DIR.exists() else []

labels = sorted(LABEL_DIR.glob("*.txt")) if LABEL_DIR.exists() else []

print("Thư mục ảnh:", IMAGE_DIR)
print("Số ảnh Test:", len(images))
print("Thư mục nhãn:", LABEL_DIR)
print("Số file nhãn:", len(labels))

print("\nMột số tên ảnh:")
for p in images[:5]:
    print(" ", p.name)

print("\nMột số file nhãn:")
for p in labels[:5]:
    print(" ", p.name)
    print("   ", p.read_text(encoding="utf-8").splitlines()[:3])

# ------------------------------------------------------------
# 2. TÌM MANIFEST VÀ BẢNG LIÊN KẾT TRONG CÁC THƯ MỤC LIÊN QUAN
# ------------------------------------------------------------

print("\n\n2. TÌM CÁC BẢNG LIÊN KẾT")
print("-" * 75)

csv_candidates = []

for search_dir in SEARCH_DIRS:
    if not search_dir.exists():
        print("[KHÔNG TÌM THẤY]", search_dir)
        continue

    for csv_path in search_dir.rglob("*.csv"):
        name = csv_path.name.lower()

        if any(key in name for key in [
            "manifest", "linkage", "dataset",
            "annotation", "split", "crop",
            "image", "label"
        ]):
            csv_candidates.append(csv_path)

for i, csv_path in enumerate(csv_candidates, start=1):
    try:
        df = pd.read_csv(csv_path, nrows=3)
        print(f"\nBẢNG {i}: {csv_path.name}")
        print("Đường dẫn:", csv_path)
        print("Kích thước mẫu:", df.shape)
        print("Các cột:", list(df.columns))
        display(df.head(3))
    except Exception as error:
        print("[LỖI ĐỌC]", csv_path, error)

# ------------------------------------------------------------
# 3. TÌM CHÍNH XÁC MANIFEST MASS CROP
# ------------------------------------------------------------

print("\n\n3. MANIFEST MASS CROP")
print("-" * 75)

crop_manifests = list(
    (ROOT / "nb08_model_training").rglob(
        "nb08_7_9_mass_crop_manifest.csv"
    )
)

if not crop_manifests:
    print("Không tìm thấy file nb08_7_9_mass_crop_manifest.csv")
else:
    for manifest_path in crop_manifests:
        print("\nĐường dẫn:", manifest_path)

        df = pd.read_csv(manifest_path)

        print("Kích thước:", df.shape)
        print("Các cột:", list(df.columns))

        # Ưu tiên các dòng thuộc Test
        split_columns = [
            c for c in df.columns
            if any(k in c.lower() for k in ["split", "set", "partition"])
        ]

        if split_columns:
            for col in split_columns:
                print(f"\nPhân bố {col}:")
                print(df[col].value_counts(dropna=False))

        display(df.head(5))

# ------------------------------------------------------------
# 4. KIỂM TRA FILE ẢNH VÀ FILE NHÃN CÓ GHÉP ĐƯỢC KHÔNG
# ------------------------------------------------------------

print("\n\n4. KIỂM TRA GHÉP ẢNH - NHÃN YOLO")
print("-" * 75)

image_stems = {p.stem for p in images}
label_stems = {p.stem for p in labels}

print("Số ảnh:", len(image_stems))
print("Số nhãn:", len(label_stems))
print("Có ảnh và nhãn cùng tên:", len(image_stems & label_stems))
print("Ảnh chưa có file nhãn:", len(image_stems - label_stems))
print("File nhãn không có ảnh:", len(label_stems - image_stems))

print("\nMột số ảnh chưa có file nhãn:")
for name in sorted(image_stems - label_stems)[:10]:
    print(" ", name)

print("\nHoàn tất truy xuất nguồn Ground Truth.")


1. KIỂM TRA ẢNH TEST VÀ NHÃN YOLO
---------------------------------------------------------------------------
Thư mục ảnh: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_3_yolo_dataset\images\test
Số ảnh Test: 361
Thư mục nhãn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_3_yolo_dataset\labels\test
Số file nhãn: 361

Một số tên ảnh:
  IMG_000001.png
  IMG_000002.png
  IMG_000003.png
  IMG_000004.png
  IMG_000005.png

Một số file nhãn:
  IMG_000001.txt
    ['0 0.81988082 0.68896655 0.18689057 0.08587119']
  IMG_000002.txt
    ['0 0.87916459 0.69632125 0.14321233 0.05991623']
  IMG_000003.txt
    ['0 0.10265625 0.65277778 0.04843750 0.02947154']
  IMG_000004.txt
    ['0 0.08934964 0.65608199 0.05519093 0.03024194']
  IMG_000005.txt
    ['0 0.77092011 0.38477326 0.12722454 0.07134480']


2. TÌM CÁC BẢNG LIÊN KẾT
---------------------------------------------------------------------------

BẢNG 1: nb06_2_full_image_dicom_audit.csv
Đường dẫn: C:\Brea

,full_image_path,annotation_count,patient_count,patient_ids,dataset_splits,...,number_of_frames,transfer_syntax_uid,sop_instance_uid,pixel_data_present,error
0,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,P_00016,test,...,1,1.2.840.10008.1.2,1.3.6.1.4.1.9590.100.1.2.255536750111234655207...,True,NaN
1,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,P_00016,test,...,1,1.2.840.10008.1.2,1.3.6.1.4.1.9590.100.1.2.270374394211018877933...,True,NaN
2,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,P_00017,test,...,1,1.2.840.10008.1.2,1.3.6.1.4.1.9590.100.1.2.294182382511340866119...,True,NaN



BẢNG 2: nb06_3_pixel_intensity_by_image.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06_detection_dataset\nb06_3_pixel_intensity_by_image.csv
Kích thước mẫu: (3, 33)
Các cột: ['full_image_path', 'annotation_count', 'patient_count', 'dataset_splits', 'image_views', 'analysis_status', 'error', 'rows', 'columns', 'pixel_count', 'pixel_dtype', 'pixel_min', 'pixel_max', 'pixel_mean', 'pixel_std', 'zero_pixel_count', 'zero_pixel_ratio', 'nonzero_pixel_count', 'nonzero_pixel_ratio', 'pixel_p1', 'nonzero_p1', 'pixel_p5', 'nonzero_p5', 'pixel_p25', 'nonzero_p25', 'pixel_p50', 'nonzero_p50', 'pixel_p75', 'nonzero_p75', 'pixel_p95', 'nonzero_p95', 'pixel_p99', 'nonzero_p99']


,full_image_path,annotation_count,patient_count,dataset_splits,image_views,...,nonzero_p75,pixel_p95,nonzero_p95,pixel_p99,nonzero_p99
0,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,test,CC,...,39867.0,41674.0,45287.0,48354.0,65535.0
1,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,test,MLO,...,48396.0,60413.0,64615.0,65535.0,65535.0
2,C:\Breast-Cancer-CBIS-DDSM\Data\rawdata\cbis_d...,1,1,test,CC,...,33290.0,35932.0,38552.0,42185.0,52489.0



BẢNG 3: nb06_3_zero_pixel_ratio_by_split.csv
Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06_detection_dataset\nb06_3_zero_pixel_ratio_by_split.csv
Kích thước mẫu: (3, 6)
Các cột: ['dataset_splits', 'image_count', 'mean_zero_ratio', 'median_zero_ratio', 'min_zero_ratio', 'max_zero_ratio']


,dataset_splits,image_count,mean_zero_ratio,median_zero_ratio,min_zero_ratio,max_zero_ratio
0,test,361,0.506563,0.506752,0.183723,0.870655
1,train,982,0.513936,0.509849,0.000065,0.859446
2,validation,249,0.511512,0.505441,0.137670,0.800190


... (47 outputs left out)

,dataset_split,total_crops,benign,malignant,patient_count,image_count
0,train,1054,546,508,552,982
1,validation,264,135,129,139,249
2,test,378,231,147,201,361




3. MANIFEST MASS CROP
---------------------------------------------------------------------------

Đường dẫn: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb08_model_training\nb08_7_classification_dataset\mass_crops_20261117_015211\nb08_7_9_mass_crop_manifest.csv
Kích thước: (1696, 22)
Các cột: ['annotation_id', 'image_id', 'patient_id', 'dataset_split', 'original_pathology', 'binary_pathology', 'source_image_path', 'crop_path', 'image_width', 'image_height', 'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max', 'crop_x_min', 'crop_y_min', 'crop_x_max', 'crop_y_max', 'crop_width', 'crop_height', 'crop_margin_ratio', 'crop_readable']

Phân bố dataset_split:
dataset_split
train         1054
test           378
validation     264
Name: count, dtype: int64


,annotation_id,image_id,patient_id,dataset_split,original_pathology,...,crop_y_max,crop_width,crop_height,crop_margin_ratio,crop_readable
0,mass_test__00001,IMG_000001,P_00016,test,MALIGNANT,...,2967,415,414,0.1,True
1,mass_test__00002,IMG_000002,P_00016,test,MALIGNANT,...,4021,346,395,0.1,True
2,mass_test__00003,IMG_000003,P_00017,test,MALIGNANT,...,3959,187,210,0.1,True
3,mass_test__00004,IMG_000004,P_00017,test,MALIGNANT,...,4013,223,216,0.1,True
4,mass_test__00005,IMG_000005,P_00032,test,BENIGN,...,2188,404,439,0.1,True




4. KIỂM TRA GHÉP ẢNH - NHÃN YOLO
---------------------------------------------------------------------------
Số ảnh: 361
Số nhãn: 361
Có ảnh và nhãn cùng tên: 361
Ảnh chưa có file nhãn: 0
File nhãn không có ảnh: 0

Một số ảnh chưa có file nhãn:

Hoàn tất truy xuất nguồn Ground Truth.


In [46]:
# 11.6 — Kiểm tra giao diện Web Demo
from pathlib import Path
from datetime import datetime
import json
import traceback

import pandas as pd
from PIL import Image

# ------------------------------------------------------------
# 1. KHỞI TẠO ĐƯỜNG DẪN
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")
WEBDEMO_DIR = PROJECT_ROOT / "WebDemo"

APP_PATH = WEBDEMO_DIR / "app.py"
RIBBON_PATH = WEBDEMO_DIR / "assets" / "breast_cancer_pink_ribbon.png"

PREPROCESSING_PATH = (
    WEBDEMO_DIR / "app" / "utils" / "image_preprocessing.py"
)
YOLO_MODULE_PATH = (
    WEBDEMO_DIR / "app" / "utils" / "yolo_detector.py"
)
INCEPTION_MODULE_PATH = (
    WEBDEMO_DIR / "app" / "utils" / "inception_classifier.py"
)

YOLO_CHECKPOINT = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_7_3_yolov5_640_finetuning"
    / "yolov5n_640_finetune_lr001_20261116_205312"
    / "weights" / "best.pt"
)

INCEPTION_CHECKPOINT = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_8_cnn_training" / "nb08_8_6_cnn_training"
    / "InceptionV3" / "best_checkpoint.pth"
)

DATASET_MANIFEST_PATH = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_3_yolo_dataset" / "dataset_manifest.csv"
)

LABEL_MANIFEST_PATH = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_3_yolo_dataset" / "label_manifest.csv"
)

CROP_MANIFEST_PATH = (
    PROJECT_ROOT
    / "Data" / "processed" / "nb08_model_training"
    / "nb08_7_classification_dataset"
    / "mass_crops_20261117_015211"
    / "nb08_7_9_mass_crop_manifest.csv"
)

REPORT_DIR = WEBDEMO_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = REPORT_DIR / "nb11_6_app_interface_report.json"

# ------------------------------------------------------------
# 2. KHỞI TẠO BÁO CÁO KIỂM TRA
# ------------------------------------------------------------

checks = {}
details = {}
errors = []


def record_check(name, condition, detail=None):
    passed = bool(condition)
    checks[name] = passed

    if detail is not None:
        details[name] = str(detail)

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}")

    if detail is not None:
        print(f"       {detail}")


def check_required_file(name, path):
    exists = path.is_file()
    size = path.stat().st_size if exists else 0

    record_check(
        name,
        exists and size > 0,
        f"{path} | Kích thước: {size:,} bytes",
    )


# ------------------------------------------------------------
# 3. KIỂM TRA CẤU TRÚC WEB DEMO
# ------------------------------------------------------------

print("\n3. KIỂM TRA CẤU TRÚC WEB DEMO")

record_check(
    "Thư mục dự án",
    PROJECT_ROOT.is_dir(),
    PROJECT_ROOT,
)

record_check(
    "Thư mục WebDemo",
    WEBDEMO_DIR.is_dir(),
    WEBDEMO_DIR,
)

record_check(
    "Thư mục app",
    (WEBDEMO_DIR / "app").is_dir(),
    WEBDEMO_DIR / "app",
)

record_check(
    "Thư mục utils",
    (WEBDEMO_DIR / "app" / "utils").is_dir(),
    WEBDEMO_DIR / "app" / "utils",
)

record_check(
    "Thư mục assets",
    (WEBDEMO_DIR / "assets").is_dir(),
    WEBDEMO_DIR / "assets",
)

record_check(
    "Thư mục config",
    (WEBDEMO_DIR / "config").is_dir(),
    WEBDEMO_DIR / "config",
)

record_check(
    "Thư mục reports",
    REPORT_DIR.is_dir(),
    REPORT_DIR,
)

record_check(
    "Thư mục outputs",
    (WEBDEMO_DIR / "outputs").is_dir(),
    WEBDEMO_DIR / "outputs",
)

# ------------------------------------------------------------
# 4. KIỂM TRA APP.PY VÀ CÁC MODULE
# ------------------------------------------------------------

print("\n4. KIỂM TRA APP.PY VÀ MODULE")

check_required_file("Ứng dụng app.py", APP_PATH)
check_required_file("Module tiền xử lý ảnh", PREPROCESSING_PATH)
check_required_file("Module YOLOv5n", YOLO_MODULE_PATH)
check_required_file("Module InceptionV3", INCEPTION_MODULE_PATH)

# Chỉ kiểm tra file tồn tại và dung lượng.
# Không chạy app.py trong Notebook và không dò chuỗi HTML/CSS.

# ------------------------------------------------------------
# 5. KIỂM TRA CHECKPOINT
# ------------------------------------------------------------

print("\n5. KIỂM TRA CHECKPOINT")

check_required_file(
    "YOLOv5n Fine-tuned checkpoint",
    YOLO_CHECKPOINT,
)

check_required_file(
    "InceptionV3 checkpoint",
    INCEPTION_CHECKPOINT,
)

# ------------------------------------------------------------
# 6. KIỂM TRA BIỂU TƯỢNG RUY-BĂNG HỒNG
# ------------------------------------------------------------

print("\n6. KIỂM TRA BIỂU TƯỢNG")

ribbon_exists = RIBBON_PATH.is_file()
ribbon_readable = False
ribbon_info = "Không tìm thấy file biểu tượng."

if ribbon_exists:
    try:
        with Image.open(RIBBON_PATH) as ribbon:
            ribbon.verify()

        with Image.open(RIBBON_PATH) as ribbon:
            ribbon_info = (
                f"Định dạng: {ribbon.format} | "
                f"Kích thước: {ribbon.size} | "
                f"Mode: {ribbon.mode}"
            )

        ribbon_readable = True

    except Exception as exc:
        ribbon_info = str(exc)

record_check(
    "File ruy-băng hồng tồn tại",
    ribbon_exists,
    RIBBON_PATH,
)

record_check(
    "File ruy-băng hồng đọc được",
    ribbon_readable,
    ribbon_info,
)

# ------------------------------------------------------------
# 7. KIỂM TRA MANIFEST
# ------------------------------------------------------------

print("\n7. KIỂM TRA MANIFEST")

check_required_file(
    "Dataset manifest",
    DATASET_MANIFEST_PATH,
)

check_required_file(
    "Label manifest",
    LABEL_MANIFEST_PATH,
)

check_required_file(
    "Mass crop manifest",
    CROP_MANIFEST_PATH,
)

# ------------------------------------------------------------
# 8. ĐỌC VÀ KIỂM TRA CẤU TRÚC DỮ LIỆU
# ------------------------------------------------------------

print("\n8. ĐỌC MANIFEST")

dataset_df = pd.read_csv(DATASET_MANIFEST_PATH)
label_df = pd.read_csv(LABEL_MANIFEST_PATH)
crop_df = pd.read_csv(CROP_MANIFEST_PATH)

required_dataset_columns = {
    "image_id",
    "image_file_name",
    "dataset_split",
    "source_image_path",
    "patient_id",
    "image_view",
    "laterality",
}

required_label_columns = {
    "annotation_id",
    "image_id",
    "dataset_split",
    "pathology",
}

required_crop_columns = {
    "annotation_id",
    "image_id",
    "patient_id",
    "dataset_split",
    "original_pathology",
    "binary_pathology",
    "source_image_path",
    "gt_x_min",
    "gt_y_min",
    "gt_x_max",
    "gt_y_max",
}

record_check(
    "Dataset manifest đủ cột",
    required_dataset_columns.issubset(dataset_df.columns),
    f"Số cột hiện có: {len(dataset_df.columns)}",
)

record_check(
    "Label manifest đủ cột",
    required_label_columns.issubset(label_df.columns),
    f"Số cột hiện có: {len(label_df.columns)}",
)

record_check(
    "Crop manifest đủ cột",
    required_crop_columns.issubset(crop_df.columns),
    f"Số cột hiện có: {len(crop_df.columns)}",
)

# ------------------------------------------------------------
# 9. LỌC TẬP TEST
# ------------------------------------------------------------

print("\n9. KIỂM TRA TẬP TEST")

dataset_df["dataset_split"] = (
    dataset_df["dataset_split"].astype(str).str.lower()
)

label_df["dataset_split"] = (
    label_df["dataset_split"].astype(str).str.lower()
)

crop_df["dataset_split"] = (
    crop_df["dataset_split"].astype(str).str.lower()
)

test_images = dataset_df[
    dataset_df["dataset_split"] == "test"
].copy()

test_labels = label_df[
    label_df["dataset_split"] == "test"
].copy()

test_crops = crop_df[
    crop_df["dataset_split"] == "test"
].copy()

test_images_unique = test_images.drop_duplicates(
    subset=["image_id"]
)

n_test_images = int(test_images_unique["image_id"].nunique())
n_test_annotations = int(test_crops["annotation_id"].nunique())
n_test_patients = int(test_crops["patient_id"].nunique())

record_check(
    "Số ảnh Test bằng 361",
    n_test_images == 361,
    f"{n_test_images:,} ảnh",
)

record_check(
    "Số annotation Test bằng 378",
    n_test_annotations == 378,
    f"{n_test_annotations:,} annotation",
)

record_check(
    "Số bệnh nhân Test bằng 201",
    n_test_patients == 201,
    f"{n_test_patients:,} bệnh nhân",
)

record_check(
    "Không trùng image_id trong danh sách ảnh Test",
    not test_images_unique["image_id"].astype(str).duplicated().any(),
    f"{len(test_images_unique):,} image_id duy nhất",
)

# ------------------------------------------------------------
# 10. KIỂM TRA PHÂN BỐ NHÃN
# ------------------------------------------------------------

print("\n10. KIỂM TRA PHÂN BỐ NHÃN")

binary_labels = (
    test_crops["binary_pathology"]
    .astype(str)
    .str.upper()
)

benign_count = int((binary_labels == "BENIGN").sum())
malignant_count = int((binary_labels == "MALIGNANT").sum())

record_check(
    "Số nhãn BENIGN bằng 231",
    benign_count == 231,
    f"{benign_count:,}",
)

record_check(
    "Số nhãn MALIGNANT bằng 147",
    malignant_count == 147,
    f"{malignant_count:,}",
)

record_check(
    "Chỉ có hai nhãn nhị phân hợp lệ",
    set(binary_labels.unique()).issubset(
        {"BENIGN", "MALIGNANT"}
    ),
    f"Các nhãn: {sorted(binary_labels.unique().tolist())}",
)

record_check(
    "Tổng nhãn bằng 378",
    benign_count + malignant_count == 378,
    f"{benign_count + malignant_count:,}",
)

# ------------------------------------------------------------
# 11. KIỂM TRA LIÊN KẾT ANNOTATION
# ------------------------------------------------------------

print("\n11. KIỂM TRA LIÊN KẾT ANNOTATION")

test_label_ids = set(
    test_labels["annotation_id"].astype(str)
)

test_crop_ids = set(
    test_crops["annotation_id"].astype(str)
)

missing_in_labels = test_crop_ids - test_label_ids
missing_in_crops = test_label_ids - test_crop_ids

record_check(
    "Annotation crop liên kết với label manifest",
    len(missing_in_labels) == 0,
    f"Annotation thiếu trong label manifest: {len(missing_in_labels)}",
)

record_check(
    "Label manifest liên kết với crop manifest",
    len(missing_in_crops) == 0,
    f"Annotation thiếu trong crop manifest: {len(missing_in_crops)}",
)

# Kiểm tra image_id của annotation phải thuộc danh sách ảnh Test.
test_image_ids = set(
    test_images_unique["image_id"].astype(str)
)

crop_image_ids = set(
    test_crops["image_id"].astype(str)
)

unmatched_image_ids = crop_image_ids - test_image_ids

record_check(
    "Annotation liên kết với ảnh Test",
    len(unmatched_image_ids) == 0,
    f"Image ID không khớp: {len(unmatched_image_ids)}",
)

# ------------------------------------------------------------
# 12. KIỂM TRA ĐƯỜNG DẪN ẢNH NGUỒN
# ------------------------------------------------------------

print("\n12. KIỂM TRA ĐƯỜNG DẪN ẢNH NGUỒN")

source_paths = (
    test_images_unique["source_image_path"]
    .dropna()
    .astype(str)
    .drop_duplicates()
)

existing_source_paths = []
missing_source_paths = []

for path_string in source_paths:
    source_path = Path(path_string)

    if source_path.is_file():
        existing_source_paths.append(source_path)
    else:
        missing_source_paths.append(path_string)

record_check(
    "Đường dẫn ảnh nguồn Test tồn tại",
    len(missing_source_paths) == 0
    and len(existing_source_paths) == 361,
    (
        f"Tồn tại: {len(existing_source_paths)}/361 | "
        f"Thiếu: {len(missing_source_paths)}"
    ),
)

# Đọc thử một ảnh nguồn để xác nhận file có thể mở.
sample_image_readable = False
sample_image_detail = "Không có ảnh nguồn để kiểm tra."

if existing_source_paths:
    try:
        with Image.open(existing_source_paths[0]) as sample_image:
            sample_image.verify()

        with Image.open(existing_source_paths[0]) as sample_image:
            sample_image_detail = (
                f"Kích thước: {sample_image.size} | "
                f"Định dạng: {sample_image.format}"
            )

        sample_image_readable = True

    except Exception as exc:
        sample_image_detail = str(exc)

record_check(
    "Ảnh nguồn Test đọc được",
    sample_image_readable,
    sample_image_detail,
)

# ------------------------------------------------------------
# 13. KIỂM TRA ĐƯỜNG DẪN ẢNH VÀ BBOX
# ------------------------------------------------------------

print("\n13. KIỂM TRA BOUNDING BOX")

bbox_columns = [
    "gt_x_min",
    "gt_y_min",
    "gt_x_max",
    "gt_y_max",
]

bbox_numeric = test_crops[bbox_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

bbox_valid = (
    bbox_numeric.notna().all(axis=1)
    & (bbox_numeric["gt_x_max"] > bbox_numeric["gt_x_min"])
    & (bbox_numeric["gt_y_max"] > bbox_numeric["gt_y_min"])
)

record_check(
    "Bounding Box Test hợp lệ",
    bool(bbox_valid.all()),
    f"Hợp lệ: {int(bbox_valid.sum())}/{len(test_crops)}",
)

# ------------------------------------------------------------
# 14. TỔNG HỢP KẾT QUẢ
# ------------------------------------------------------------

print("\n14. TỔNG KẾT NOTEBOOK 11.6")

passed_count = sum(checks.values())
failed_count = len(checks) - passed_count

overall_status = (
    "PASS" if failed_count == 0 else "CHECK_REQUIRED"
)

report = {
    "notebook": "11.6",
    "title": "Kiểm tra giao diện Web Demo",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "webdemo_dir": str(WEBDEMO_DIR),
    "app_path": str(APP_PATH),
    "report_path": str(REPORT_PATH),
    "test_dataset": {
        "images": int(n_test_images),
        "annotations": int(n_test_annotations),
        "patients": int(n_test_patients),
        "benign": int(benign_count),
        "malignant": int(malignant_count),
    },
    "source_images": {
        "existing": int(len(existing_source_paths)),
        "missing": int(len(missing_source_paths)),
    },
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "details": details,
    "summary": {
        "total_checks": int(len(checks)),
        "passed": int(passed_count),
        "failed": int(failed_count),
        "status": overall_status,
    },
}

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        report,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"\n[INFO] Báo cáo: {REPORT_PATH}")
print(f"[INFO] Tổng kiểm tra: {len(checks)}")
print(f"[PASS] Thành công: {passed_count}")
print(f"[FAIL] Cần kiểm tra: {failed_count}")
print(f"[STATE] {overall_status}")

if overall_status == "PASS":
    print("\n[PASS] NOTEBOOK 11.6 HOÀN TẤT")
else:
    print("\n[WARNING] NOTEBOOK 11.6 CÒN HẠNG MỤC CẦN KIỂM TRA")



3. KIỂM TRA CẤU TRÚC WEB DEMO
[PASS] Thư mục dự án
       C:\Breast-Cancer-CBIS-DDSM
[PASS] Thư mục WebDemo
       C:\Breast-Cancer-CBIS-DDSM\WebDemo
[PASS] Thư mục app
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\app
[PASS] Thư mục utils
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils
[PASS] Thư mục assets
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\assets
[PASS] Thư mục config
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\config
[PASS] Thư mục reports
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\reports
[PASS] Thư mục outputs
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\outputs

4. KIỂM TRA APP.PY VÀ MODULE
[PASS] Ứng dụng app.py
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\app.py | Kích thước: 56,896 bytes
[PASS] Module tiền xử lý ảnh
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils\image_preprocessing.py | Kích thước: 7,762 bytes
[PASS] Module YOLOv5n
       C:\Breast-Cancer-CBIS-DDSM\WebDemo\app\utils\yolo_detector.py | Kích thước: 6,486 bytes
[PASS] Module InceptionV3
       C:\Breast-Canc